In [3]:
import sqlite3
import pandas as pd
from pathlib import Path

# Locate the database
paths = [
    Path("../backend/database/cinematch.db")
]

DB_PATH = next((p.resolve() for p in paths if p.exists()), None)

if DB_PATH is None:
    raise FileNotFoundError("Could not locate cinematch.db")

print("Database found:", DB_PATH)

Database found: /Users/pragya.sharma/Documents/CPSC481_Final/backend/database/cinematch.db


In [4]:
# collecting each movie's details along with its tags in a single column

conn = sqlite3.connect(DB_PATH)

query = """
SELECT
    m.movie_id,
    m.title,
    m.year,
    m.genres,
    COALESCE(GROUP_CONCAT(t.tag, ' '), '') AS tags
FROM movies m
LEFT JOIN tags t ON m.movie_id = t.movie_id
GROUP BY m.movie_id
ORDER BY m.movie_id;
"""

movies_df = pd.read_sql_query(query, conn)

conn.close()

movies_df.head()

,movie_id,title,year,genres,tags
0,1,Toy Story (1995),1995.0,Adventure|Animation|Children|Comedy|Fantasy,fun pixar pixar
1,2,Jumanji (1995),1995.0,Adventure|Children|Fantasy,fantasy game magic board game robin williams
2,3,Grumpier Old Men (1995),1995.0,Comedy|Romance,moldy old
3,4,Waiting to Exhale (1995),1995.0,Comedy|Drama|Romance,
4,5,Father of the Bride Part II (1995),1995.0,Comedy,pregnancy remake


In [5]:
# running the extracted data to check if the data is correct and the number of rows and columns are correct

print("Number of movies:", len(movies_df))
print("Columns:", movies_df.columns.tolist())

movies_df.head(15)

Number of movies: 9742
Columns: ['movie_id', 'title', 'year', 'genres', 'tags']


,movie_id,title,year,genres,tags
0,1,Toy Story (1995),1995.0,Adventure|Animation|Children|Comedy|Fantasy,fun pixar pixar
1,2,Jumanji (1995),1995.0,Adventure|Children|Fantasy,fantasy game magic board game robin williams
2,3,Grumpier Old Men (1995),1995.0,Comedy|Romance,moldy old
3,4,Waiting to Exhale (1995),1995.0,Comedy|Drama|Romance,
4,5,Father of the Bride Part II (1995),1995.0,Comedy,pregnancy remake
5,6,Heat (1995),1995.0,Action|Crime|Thriller,
6,7,Sabrina (1995),1995.0,Comedy|Romance,remake
7,8,Tom and Huck (1995),1995.0,Adventure|Children,
8,9,Sudden Death (1995),1995.0,Action,
9,10,GoldenEye (1995),1995.0,Action|Adventure|Thriller,


In [6]:
# Handle missing values (shouldn't be any, but just in case)
movies_df["genres"] = movies_df["genres"].fillna("")
movies_df["tags"] = movies_df["tags"].fillna("")

# Clean tags
movies_df["tags_clean"] = (
    movies_df["tags"]
    .str.lower()
    .str.strip()
)

# Convert genres from "|" separated format to spaces
movies_df["genres_clean"] = (
    movies_df["genres"]
    .str.replace("|", " ", regex=False)
    .str.lower()
    .str.strip()
)

movies_df[["title", "genres_clean", "tags_clean"]].head(15)

,title,genres_clean,tags_clean
0,Toy Story (1995),adventure animation children comedy fantasy,fun pixar pixar
1,Jumanji (1995),adventure children fantasy,fantasy game magic board game robin williams
2,Grumpier Old Men (1995),comedy romance,moldy old
3,Waiting to Exhale (1995),comedy drama romance,
4,Father of the Bride Part II (1995),comedy,pregnancy remake
5,Heat (1995),action crime thriller,
6,Sabrina (1995),comedy romance,remake
7,Tom and Huck (1995),adventure children,
8,Sudden Death (1995),action,
9,GoldenEye (1995),action adventure thriller,


In [7]:
# combining the genres and tags into a single column for each movie to create a movie profile

movies_df["movie_profile"] = (
    movies_df["genres_clean"] + " " + movies_df["tags_clean"]
).str.strip()

movies_df[["title", "movie_profile"]].head(10)

,title,movie_profile
0,Toy Story (1995),adventure animation children comedy fantasy fu...
1,Jumanji (1995),adventure children fantasy fantasy game magic ...
2,Grumpier Old Men (1995),comedy romance moldy old
3,Waiting to Exhale (1995),comedy drama romance
4,Father of the Bride Part II (1995),comedy pregnancy remake
5,Heat (1995),action crime thriller
6,Sabrina (1995),comedy romance remake
7,Tom and Huck (1995),adventure children
8,Sudden Death (1995),action
9,GoldenEye (1995),action adventure thriller


In [8]:
# Check for missing profiles (shouldn't be any, but just in case)
print("Missing profiles:", movies_df["movie_profile"].isna().sum())

# Check for empty profiles
empty_profiles = movies_df["movie_profile"].str.strip().eq("").sum()
print("Empty profiles:", empty_profiles)

# Inspect a random sample
movies_df[["title", "movie_profile"]].sample(10, random_state=42)

Missing profiles: 0
Empty profiles: 0


,title,movie_profile
6213,"Notorious Bettie Page, The (2005)",drama
1056,Star Trek V: The Final Frontier (1989),action sci-fi
4891,Starsky & Hutch (2004),action comedy crime thriller
346,"Colonel Chabert, Le (1994)",drama romance war
5353,"Country Girl, The (1954)",drama oscar (best actress)
7829,"Civil War, The (1990)",documentary war
8655,John Mulaney: New In Town (2012),comedy
9056,Saving Christmas (2014),children comedy
1111,Kolya (Kolja) (1996),comedy drama in netflix queue
7278,Alice in Wonderland (2010),adventure fantasy imax


In [9]:
import numpy as np
from scipy.sparse import hstack
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import normalize

movies_df = movies_df.reset_index(drop=True)

# Genres: keep each genre as one token (e.g. "sci-fi" stays intact)
movies_df["genres_tok"] = (
    movies_df["genres"].str.lower().str.replace(" ", "", regex=False)
)
genre_vec = TfidfVectorizer(tokenizer=lambda s: s.split("|"), token_pattern=None)
genre_matrix = genre_vec.fit_transform(movies_df["genres_tok"])

# Tags: unigrams only (or bigrams with min_df), sublinear TF
tag_vec = TfidfVectorizer(
    stop_words="english",
    ngram_range=(1, 2),
    min_df=3,
    sublinear_tf=True
)
tag_matrix = tag_vec.fit_transform(movies_df["tags_clean"])

# Combine with weights (tune these)
w_genre, w_tags = 0.4, 0.6
tfidf_matrix = normalize(hstack([w_genre * genre_matrix, w_tags * tag_matrix]).tocsr())

feature_names = np.concatenate([
    genre_vec.get_feature_names_out(),
    tag_vec.get_feature_names_out()
])

In [14]:
print("Number of movies:", tfidf_matrix.shape[0])
print("Number of unique features:", tfidf_matrix.shape[1])

Number of movies: 9742
Number of unique features: 604


In [15]:
print("First 100 features:")
print(feature_names[:100])

First 100 features:
['(nogenreslisted)' 'action' 'adventure' 'animation' 'children' 'comedy'
 'crime' 'documentary' 'drama' 'fantasy' 'film-noir' 'horror' 'imax'
 'musical' 'mystery' 'romance' 'sci-fi' 'thriller' 'war' 'western' '06'
 '06 oscar' '1970s' '250' 'aardman' 'abuse' 'acting' 'action' 'actress'
 'adam' 'adam sandler' 'adolescence' 'adultery' 'adventure' 'affleck'
 'africa' 'al' 'al pacino' 'alcoholism' 'alfred' 'alfred hitchcock'
 'aliens' 'alternate' 'alternate reality' 'amazing' 'america' 'american'
 'amnesia' 'andy' 'animal' 'animal movie' 'animation' 'animation anime'
 'anime' 'anne' 'anti' 'anti semitism' 'apocalyptic' 'appealing'
 'appealing visually' 'archaeology' 'art' 'arthur' 'artificial'
 'artificial intelligence' 'artistic' 'arts' 'artsy' 'artsy atmospheric'
 'assassin' 'assassination' 'astaire' 'astaire rogers' 'atmospheric'
 'atmospheric atmospheric' 'atmospheric beautiful' 'atmospheric cerebral'
 'atmospheric creepy' 'austen' 'australia' 'awesome' 'bad' 'bad pl

In [16]:
import numpy as np

movie_title = "Toy Story (1995)"

movie_index = movies_df.index[
    movies_df["title"] == movie_title
][0]

movie_vector = tfidf_matrix[movie_index]

# Extract nonzero features and their weights
feature_indices = movie_vector.nonzero()[1]
feature_weights = movie_vector.data

movie_features = pd.DataFrame({
    "feature": feature_names[feature_indices],
    "tfidf_weight": feature_weights
}).sort_values("tfidf_weight", ascending=False)

movie_features.head(15)

,feature,tfidf_weight
6,pixar,0.723662
5,fun,0.410635
1,animation,0.286350
2,children,0.280038
4,fantasy,0.267915
0,adventure,0.231224
3,comedy,0.148430


In [17]:
from sklearn.metrics.pairwise import cosine_similarity

# Select a random movie
movie_title = "Friday (1995)"

movie_index = movies_df.index[
    movies_df["title"] == movie_title
][0]

# Compare this movie against every movie in the dataset
similarity_scores = cosine_similarity(
    tfidf_matrix,
    tfidf_matrix[movie_index]
).flatten()

print("Number of similarity scores:", len(similarity_scores))

Number of similarity scores: 9742


In [18]:
# Exclude the selected movie itself
similarity_scores[movie_index] = -1

# Sort movies by similarity score (highest to lowest)
similar_movies_indices = np.argsort(similarity_scores)[::-1]

# Select the top 20
top_20_indices = similar_movies_indices[:20]

# Display recommendations
recommendations = movies_df.iloc[top_20_indices][
    ["movie_id", "title", "genres", "tags"]
].copy()

recommendations["similarity_score"] = similarity_scores[top_20_indices]

recommendations

,movie_id,title,genres,tags,similarity_score
9741,193609,Andrew Dice Clay: Dice Rules (1991),Comedy,,1.0
3332,4511,High Spirits (1988),Comedy,,1.0
3327,4504,Feds (1988),Comedy,,1.0
3324,4499,Dirty Rotten Scoundrels (1988),Comedy,,1.0
3318,4490,"Couch Trip, The (1988)",Comedy,,1.0
9050,141836,It Can't Be! (1975),Comedy,,1.0
9048,141820,Old Men: Robbers (1971),Comedy,,1.0
7705,89939,Gigi (1949),Comedy,,1.0
3313,4483,Caddyshack II (1988),Comedy,,1.0
3292,4452,Made (2001),Comedy,,1.0


In [19]:
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

def recommend_similar_movies(movie_title, top_n=10):
    # Search for movies matching the title (case-insensitive)
    matches = movies_df[
        movies_df["title"].str.contains(
            movie_title, case=False, na=False, regex=False
        )
    ]

    # Check for no matches BEFORE trying to use them
    if matches.empty:
        print(f"No movie found matching '{movie_title}'")
        return None

    # Prefer an exact title match, otherwise use the first partial match
    exact_matches = matches[
        matches["title"].str.lower() == movie_title.lower()
    ]
    chosen = exact_matches if not exact_matches.empty else matches

    # Convert the index label to a positional index so it lines up
    # with tfidf_matrix and iloc, even if movies_df has a custom index
    movie_pos = movies_df.index.get_loc(chosen.index[0])

    selected_movie = movies_df.iloc[movie_pos]

    print(f"Recommendations for: {selected_movie['title']}")

    # Cosine similarity of the selected movie against every movie
    scores = cosine_similarity(
        tfidf_matrix[movie_pos], tfidf_matrix
    ).flatten()

    # Exclude the selected movie itself
    scores[movie_pos] = -np.inf

    # Top N most similar (guard against top_n > number of movies)
    top_n = min(top_n, len(scores) - 1)
    top_indices = np.argsort(scores)[::-1][:top_n]

    recommendations = movies_df.iloc[top_indices][
        ["movie_id", "title", "genres", "tags"]
    ].copy()
    recommendations["similarity_score"] = scores[top_indices]
    recommendations = recommendations.reset_index(drop=True)

    display(recommendations)  # use print(recommendations) outside Jupyter

    return recommendations

In [20]:
recommend_similar_movies("Titanic", 10)

Recommendations for: Titanic (1997)


,movie_id,title,genres,tags,similarity_score
0,351,"Corrina, Corrina (1994)",Comedy|Drama|Romance,interracial romance,0.957880
1,1235,Harold and Maude (1971),Comedy|Drama|Romance,may-december romance,0.957880
2,8533,"Notebook, The (2004)",Drama|Romance,memory loss romance sad,0.687445
3,8921,"Rose Tattoo, The (1955)",Drama|Romance,,0.554700
4,1290,Some Kind of Wonderful (1987),Drama|Romance,,0.554700
5,147002,Eros (2004),Drama|Romance,,0.554700
6,172233,The Prime Gig (2000),Drama|Romance,,0.554700
7,25833,Camille (1936),Drama|Romance,,0.554700
8,7038,Things You Can Tell Just by Looking at Her (2000),Drama|Romance,,0.554700
9,4041,"Officer and a Gentleman, An (1982)",Drama|Romance,,0.554700


,movie_id,title,genres,tags,similarity_score
0,351,"Corrina, Corrina (1994)",Comedy|Drama|Romance,interracial romance,0.957880
1,1235,Harold and Maude (1971),Comedy|Drama|Romance,may-december romance,0.957880
2,8533,"Notebook, The (2004)",Drama|Romance,memory loss romance sad,0.687445
3,8921,"Rose Tattoo, The (1955)",Drama|Romance,,0.554700
4,1290,Some Kind of Wonderful (1987),Drama|Romance,,0.554700
5,147002,Eros (2004),Drama|Romance,,0.554700
6,172233,The Prime Gig (2000),Drama|Romance,,0.554700
7,25833,Camille (1936),Drama|Romance,,0.554700
8,7038,Things You Can Tell Just by Looking at Her (2000),Drama|Romance,,0.554700
9,4041,"Officer and a Gentleman, An (1982)",Drama|Romance,,0.554700
